# 📓 Notebook 2｜常態分類器：LDA/QDA 與馬氏距離（Example 2.2）

> 對應講義 Part 3 ＋ Demo 2。驗證課本 Example 2.2 的馬氏距離判別。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 課本 Example 2.2：共享共變異數矩陣、均值 [0,0]ᵀ 與 [3,3]ᵀ
SIG = np.array([[1.1, 0.3], [0.3, 1.9]])
mu1, mu2 = np.array([0.0, 0.0]), np.array([3.0, 3.0])
x = np.array([1.0, 2.2])

Sinv = np.linalg.inv(SIG)
def mahal(mu_, x_): return float((x_-mu_) @ Sinv @ (x_-mu_))
d1, d2 = mahal(mu1, x), mahal(mu2, x)
de1, de2 = np.linalg.norm(x-mu1), np.linalg.norm(x-mu2)
print(f'馬氏距離²：d₁²={d1:.3f}  d₂²={d2:.3f}  → 分到 ω1（與課本 2.952/3.672 一致）')
print(f'歐氏距離 ：d₁={de1:.3f} d₂={de2:.3f}  → 歐氏反而較近 ω2（馬氏才對）！')

# 主軸（特徵分解）
w, V = np.linalg.eigh(SIG)
print('特徵值 λ =', np.round(w, 4), ' 特徵向量 =\n', np.round(V, 4))

In [ ]:
# 貝氏決策面（QDA vs LDA）：網格分類繪圖
def bayes_class(LDA=True, N=401):
    xg = np.linspace(-3, 6, N); yy, xx = np.meshgrid(xg, xg)
    pts = np.stack([xx.ravel(), yy.ravel()], 1)
    if LDA:  # Σ 相同 → 二次項抵消
        w_ = Sinv @ (mu2 - mu1)
        g = pts @ w_ + ( -0.5*(mu2@Sinv@mu2 - mu1@Sinv@mu1))
        cls = (g > 0).astype(int)
    else:
        S1 = SIG; S2 = SIG.copy(); S2[0,0]*=2.2   # ω2 方差較大
        def dd(p, m, Si): return ((p-m) @ np.linalg.inv(Si) * (p-m)).sum(1) + np.log(np.linalg.det(Si))
        cls = (dd(pts, mu1, S1) > dd(pts, mu2, S2)).astype(int)
    return xx, yy, cls.reshape(N, N)

for tag, LDA in [('LDA（共享 Σ → 直線）', True), ('QDA（Σ₂ 較大 → 曲線）', False)]:
    xx, yy, cls = bayes_class(LDA)
    plt.figure(figsize=(5, 4.5))
    plt.contourf(xx, yy, cls, levels=[-0.5, 0.5, 1.5], alpha=0.35, colors=['#2563eb', '#dc2626'])
    plt.scatter([mu1[0]], [mu1[1]], s=90, c='#1d4ed8', marker='x'); plt.scatter([mu2[0]], [mu2[1]], s=90, c='#b91c1c', marker='x')
    plt.scatter([x[0]], [x[1]], s=140, c='k', marker='*')
    plt.xlim(-3, 6); plt.ylim(-3, 6); plt.title(tag); plt.xlabel('x1'); plt.ylabel('x2')
    plt.tight_layout(); plt.show()
print('星號 = 待分類點 [1.0, 2.2]ᵀ：LDA 下屬 ω1（藍）區域 ✅')

### ✏️ 練習
1. 把 `x` 改成 [1.8, 2.2]ᵀ：歐氏/馬氏仍一致嗎？（課本提醒：靠近哪個均值不一定等於近哪個類）
2. 改 QDA 分支裡的 `S2`，觀察決策面從直線變曲線的角度。